In [1]:
from dotenv import load_dotenv
load_dotenv()  # loads from .env in cwd or parent dirs
import wandb


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

api = wandb.Api()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


In [47]:
from collections import defaultdict

PROJECT_PATH = "jdm8943-rochester-institute-of-technology/style-prompt-gen"

runs = api.runs(PROJECT_PATH)

rows_results  = []
rows_baseline = []

for run in runs:
    if run.state != "finished":
        continue

    datasource_data = defaultdict(dict)
    baseline_data   = defaultdict(dict)
    trial_metrics   = {}

    for k, v in run.summary.items():
        if k.startswith("_"):
            continue
        parts = k.split("/")
        if k.startswith("test/"):
            datasource_data[parts[1]][parts[2]] = v
        elif k.startswith("trial/"):
            trial_metrics[parts[1]] = v
        elif k.startswith("baseline/"):
            baseline_data[parts[1]][parts[2]] = v

    for datasource, metrics in datasource_data.items():
        rows_results.append({
            "run_name":        run.name,
            "test_datasource": datasource,
            "experiment_param": run.config.get("experiment_param"),
            "experiment_value": run.config.get("experiment_value"),
            **trial_metrics,
            **metrics,
        })

    for datasource, metrics in baseline_data.items():
        rows_baseline.append({
            "run_name":        run.name,
            "test_datasource": datasource,
            "experiment_param": run.config.get("experiment_param"),
            "experiment_value": run.config.get("experiment_value"),
            **metrics,
        })

results_df  = pd.DataFrame(rows_results).dropna(subset=["experiment_param"])
baseline_df = pd.DataFrame(rows_baseline).dropna(subset=["experiment_param"])

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [42]:
results_df

,run_name,test_datasource,experiment_param,experiment_value,trial_idx,training_time_s,bertscore_f1,bertscore_f1_std,bertscore_precision,bertscore_precision_std,bertscore_recall,bertscore_recall_std,chrf,chrf_std,dist1,dist2,inference_time_s,mean_cosine_sim,meteor,meteor_std,pred_semantic_sim,rougeL,rougeL_std,tag_f1_accent,tag_f1_accent_std,tag_f1_gender,tag_f1_gender_std,tag_f1_intrinsic_tags,tag_f1_intrinsic_tags_std,tag_f1_noise,tag_f1_noise_std,tag_f1_overall,tag_f1_overall_std,tag_f1_pitch,tag_f1_pitch_std,tag_f1_situational_tags,tag_f1_situational_tags_std,tag_f1_speaking_rate,tag_f1_speaking_rate_std,vec_norm_cv,vec_std,tag_f1_emotion,tag_f1_emotion_std,tag_f1_volume,tag_f1_volume_std
0,worldly-pyramid-43,expresso,num_turns,1,0,2312.056786,0.918890,0.015036,0.917404,0.013862,0.920496,0.019117,0.492544,0.098704,0.025774,0.078181,47.865992,0.637153,0.388508,0.124031,0.817217,0.381787,0.090056,0.895,0.306553,0.610000,0.487750,0.447307,0.311640,0.485000,0.499775,0.568377,0.207472,0.925,0.263391,0.251333,0.327140,0.413333,0.491302,0.000107,0.499536,NaN,NaN,NaN,NaN
1,worldly-pyramid-43,styletalk,num_turns,1,0,2312.056786,0.977552,0.017968,0.977005,0.020938,0.978143,0.015567,0.811432,0.130472,0.021413,0.043701,16.484042,0.791805,0.767510,0.133349,0.924567,0.731998,0.125646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.687500,0.292944,NaN,NaN,NaN,NaN,0.705000,0.456043,0.000102,0.370252,0.535,0.498773,0.820,0.384187
2,dandy-morning-46,expresso,num_turns,1,0,1967.845897,0.918890,0.015036,0.917404,0.013862,0.920496,0.019117,0.492544,0.098704,0.025774,0.078181,40.953415,0.637153,0.388508,0.124031,0.817217,0.381787,0.090056,0.895,0.306553,0.610000,0.487750,0.447307,0.311640,0.485000,0.499775,0.568377,0.207472,0.925,0.263391,0.251333,0.327140,0.413333,0.491302,0.000107,0.499536,NaN,NaN,NaN,NaN
3,dandy-morning-46,styletalk,num_turns,1,0,1967.845897,0.977552,0.017968,0.977005,0.020938,0.978143,0.015567,0.811432,0.130472,0.021413,0.043701,13.021299,0.791805,0.767510,0.133349,0.924567,0.731998,0.125646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.687500,0.292944,NaN,NaN,NaN,NaN,0.705000,0.456043,0.000102,0.370252,0.535,0.498773,0.820,0.384187
4,gentle-dust-51,expresso,num_mapping_layers,2,0,247.903737,0.919965,0.009611,0.918729,0.008620,0.921290,0.013809,0.482120,0.066805,0.006955,0.012368,39.451600,0.951361,0.356565,0.097499,0.983024,0.389495,0.067964,0.165,0.371181,0.530000,0.499099,0.424945,0.201864,0.515000,0.499775,0.491589,0.139693,0.960,0.195959,0.299333,0.337474,0.395000,0.488851,0.062325,0.190151,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,true-cosmos-168,styletalk,num_prefix_tokens,20,4,3377.204839,0.920816,0.007360,0.897430,0.006378,0.945527,0.011873,0.580711,0.072863,0.005240,0.008369,23.228471,0.984259,0.603331,0.093675,0.964517,0.458717,0.049693,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.603333,0.277669,NaN,NaN,NaN,NaN,0.710000,0.453762,0.000143,0.106229,0.265,0.441333,0.835,0.371181
240,distinctive-eon-169,expresso,num_mapping_layers,8,4,3372.479858,0.923325,0.010624,0.924249,0.010108,0.922493,0.014411,0.500604,0.071385,0.007761,0.013240,53.352986,0.965546,0.397383,0.106845,0.924736,0.394698,0.080233,0.955,0.207304,0.550000,0.497494,0.434853,0.195585,0.515000,0.499775,0.555249,0.123914,0.960,0.195959,0.225667,0.294500,0.380000,0.485386,0.029541,0.168220,NaN,NaN,NaN,NaN
241,distinctive-eon-169,styletalk,num_mapping_layers,8,4,3372.479858,0.917472,0.016329,0.884316,0.023769,0.953436,0.011806,0.613065,0.089061,0.003803,0.005073,23.120686,0.975955,0.563350,0.104126,0.964800,0.405018,0.092909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.590000,0.274287,NaN,NaN,NaN,NaN,0.710000,0.453762,0.000280,0.134772,0.225,0.417582,0.835,0.371181
242,pleasant-pine-170,expresso,num_unfrozen_embedder_layers,2,4,3182.366340,0.919108,0.012074,0.916812,0.012633,0.921512,0.014904,0.497137,0.076157,0.033728,0.125264,53.317256,0.66224

In [43]:
baseline_df

,run_name,test_datasource,experiment_param,experiment_value,trial_idx,bertscore_f1,bertscore_f1_std,bertscore_precision,bertscore_precision_std,bertscore_recall,bertscore_recall_std,chrf,chrf_std,dist1,dist2,inference_time_s,meteor,meteor_std,pred_semantic_sim,rougeL,rougeL_std,tag_f1_accent,tag_f1_accent_std,tag_f1_gender,tag_f1_gender_std,tag_f1_intrinsic_tags,tag_f1_intrinsic_tags_std,tag_f1_noise,tag_f1_noise_std,tag_f1_overall,tag_f1_overall_std,tag_f1_pitch,tag_f1_pitch_std,tag_f1_situational_tags,tag_f1_situational_tags_std,tag_f1_speaking_rate,tag_f1_speaking_rate_std,tag_f1_emotion,tag_f1_emotion_std,tag_f1_volume,tag_f1_volume_std
0,worldly-pyramid-43,expresso,num_turns,1,0,0.862779,0.030729,0.863442,0.032139,0.862295,0.031664,0.259335,0.120530,0.127310,0.361931,88.238232,0.139161,0.109088,0.506681,0.172938,0.111706,0.365,0.481430,0.140000,0.346987,0.140210,0.260029,0.40,0.489898,0.204438,0.237415,0.400,0.489898,0.131619,0.302664,0.380,0.485386,NaN,NaN,NaN,NaN
1,worldly-pyramid-43,styletalk,num_turns,1,0,0.866744,0.023118,0.852789,0.022241,0.881383,0.027769,0.233521,0.095743,0.116945,0.346128,87.245789,0.161718,0.114297,0.498410,0.172395,0.091563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.060167,0.184863,NaN,NaN,NaN,NaN,0.050,0.217945,0.035000,0.183780,0.060,0.237487
2,dandy-morning-46,expresso,num_turns,1,0,0.862779,0.030729,0.863442,0.032139,0.862295,0.031664,0.259335,0.120530,0.127310,0.361931,80.194030,0.139161,0.109088,0.506681,0.172938,0.111706,0.365,0.481430,0.140000,0.346987,0.140210,0.260029,0.40,0.489898,0.204438,0.237415,0.400,0.489898,0.131619,0.302664,0.380,0.485386,NaN,NaN,NaN,NaN
3,dandy-morning-46,styletalk,num_turns,1,0,0.866744,0.023118,0.852789,0.022241,0.881383,0.027769,0.233521,0.095743,0.116945,0.346128,78.504387,0.161718,0.114297,0.498410,0.172395,0.091563,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.060167,0.184863,NaN,NaN,NaN,NaN,0.050,0.217945,0.035000,0.183780,0.060,0.237487
4,gentle-dust-51,expresso,num_mapping_layers,2,0,0.896351,0.022916,0.895352,0.022310,0.897469,0.025565,0.398257,0.111549,0.090430,0.313251,170.165325,0.263675,0.123123,0.656824,0.300104,0.102183,0.740,0.438634,0.528333,0.498082,0.383090,0.328700,0.45,0.497494,0.535240,0.237031,0.815,0.388298,0.424405,0.408860,0.400,0.489898,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,true-cosmos-168,styletalk,num_prefix_tokens,20,4,0.887117,0.018691,0.872625,0.019194,0.902218,0.020857,0.297629,0.095835,0.077127,0.281352,163.832436,0.225113,0.117141,0.693579,0.241191,0.095719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.151833,0.256833,NaN,NaN,NaN,NaN,0.150,0.357071,0.120000,0.324962,0.090,0.286182
240,distinctive-eon-169,expresso,num_mapping_layers,8,4,0.905504,0.015014,0.904754,0.014202,0.906360,0.018542,0.429057,0.086419,0.070526,0.266394,175.351517,0.299890,0.104876,0.751494,0.323506,0.076952,0.820,0.384187,0.655000,0.475368,0.429407,0.320834,0.38,0.485386,0.595317,0.191223,0.895,0.306553,0.528571,0.415362,0.395,0.488851,NaN,NaN,NaN,NaN
241,distinctive-eon-169,styletalk,num_mapping_layers,8,4,0.890036,0.019020,0.875682,0.019601,0.904999,0.021315,0.309329,0.112753,0.076096,0.267750,167.264338,0.243645,0.148190,0.706997,0.245778,0.109313,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.148167,0.254988,NaN,NaN,NaN,NaN,0.165,0.371181,0.115000,0.319022,0.080,0.271293
242,pleasant-pine-170,expresso,num_unfrozen_embedder_layers,2,4,0.904750,0.018539,0.904418,0.017854,0.905181,0.021401,0.428849,0.099032,0.071187,0.276425,171.486398,0.292716,0.114543,0.732902,0.328358,0.095569,0.770,0.420833,0.625000,0.484123,0.416730,0.306680,0.46,0.498397,0.579411,0.215290,0.930,0.255147,0.480857,0.413961,0.350,0.476970,NaN,NaN,NaN,NaN


In [48]:
def summarize(df, group_cols):
    drop_std_cols = [c for c in df.columns if c.endswith("_std")]
    agg_df = df.drop(columns=["run_name"] + drop_std_cols, errors="ignore")
    return agg_df.groupby(group_cols).agg(["mean", "std"]).round(4)

results_summary  = summarize(results_df,  ["experiment_param", "experiment_value", "test_datasource"])
baseline_summary = summarize(baseline_df, ["experiment_param", "experiment_value", "test_datasource"])


In [49]:
results_summary

training_time_s  \
                                                                         mean   
experiment_param             experiment_value test_datasource                   
num_mapping_layers           2                expresso              1709.7952   
                                              styletalk             1709.7952   
                             4                expresso              1810.6611   
                                              styletalk             1810.6611   
                             8                expresso              1818.2024   
                                              styletalk             1818.2024   
num_prefix_tokens            5                expresso              1753.7155   
                                              styletalk             1753.7155   
                             10               expresso              1756.4923   
                                              styletalk             1756.4923   
                             20               expresso              1816.7852   
                                              styletalk             1816.7852   
num_turns                    1                expresso              1233.7190   
                                              styletalk             1233.7190   
                             3                expresso              1690.8952   
                                              styletalk             1690.8952   
                             5                expresso              1868.8863   
                                              styletalk             1868.8863   
num_unfrozen_embedder_layers 0                expresso              1749.6570   
                                              styletalk             1749.6570   
                             1                expresso              1746.6983   
                                              styletalk             1746.6983   
                             2                expresso              1715.9722   
                                              styletalk             1715.9722   

                                                                          \
                                                                     std   
experiment_param             experiment_value test_datasource              
num_mapping_layers           2                expresso         1543.5901   
                                              styletalk        1543.5901   
                             4                expresso         1645.7611   
                                              styletalk        1645.7611   
                             8                expresso         1646.2289   
                                              styletalk        1646.2289   
num_prefix_tokens            5                expresso         1589.2790   
                                              styletalk        1589.2790   
                             10               expresso         1592.8670   
                                              styletalk        1592.8670   
                             20               expresso         1654.2342   
                                              styletalk        1654.2342   
num_turns                    1                expresso          980.8352   
                                              styletalk         980.8352   
                             3                expresso         1561.2940   
                                              styletalk        1561.2940   
                             5                expresso         1727.0311   
                                              styletalk        1727.0311   
num_unfrozen_embedder_layers 0                expresso         1615.6340   
                                              styletalk        1615.6340   
                             1                expresso         1612.8962   
                                              styl

In [ ]:
baseline_summary